In [ ]:
from collections import defaultdict, Counter

class KneserNey5Gram:

    def __init__(self, discount=0.75):
        self.discount = discount

        # Stores n-gram counts
        self.ngram_counts = {
            1: Counter(),
            2: Counter(),
            3: Counter(),
            4: Counter(),
            5: Counter()
        }

        # Context counts
        self.context_counts = {
            1: Counter(),
            2: Counter(),
            3: Counter(),
            4: Counter()
        }

        # Continuation sets
        self.continuation = defaultdict(set)

        self.vocab = set()

    def tokenize(self, sentence):
        return sentence.lower().split()

    def train(self, corpus):

        for sentence in corpus:

            words = ["<s>"] * 4 + self.tokenize(sentence) + ["</s>"]

            self.vocab.update(words)

            for n in range(1,6):

                for i in range(len(words)-n+1):

                    gram = tuple(words[i:i+n])

                    self.ngram_counts[n][gram] += 1

                    if n > 1:
                        context = gram[:-1]
                        self.context_counts[n-1][context] += 1
                        self.continuation[gram[-1]].add(context)

    def continuation_probability(self, word):

        if word not in self.vocab:
            return 1 / (len(self.vocab)+1)

        return len(self.continuation[word]) / max(1,len(self.ngram_counts[2]))

    def probability(self, ngram):

        n = len(ngram)

        if n == 1:
            return self.continuation_probability(ngram[0])

        context = ngram[:-1]

        count_ngram = self.ngram_counts[n][ngram]
        count_context = self.context_counts[n-1][context]

        if count_context == 0:
            return self.probability(ngram[1:])

        unique_followers = 0

        for g in self.ngram_counts[n]:
            if g[:-1] == context:
                unique_followers += 1

        lambda_weight = (self.discount * unique_followers) / count_context

        discounted = max(count_ngram - self.discount,0) / count_context

        backoff = self.probability(ngram[1:])

        return discounted + lambda_weight * backoff

    def predict_next(self, text, topk=5):

        tokens = self.tokenize(text)

        tokens = ["<s>"] * max(0,4-len(tokens)) + tokens

        context = tuple(tokens[-4:])

        scores = []

        for word in self.vocab:

            if word == "<s>":
                continue

            gram = context + (word,)

            prob = self.probability(gram)

            scores.append((word,prob))

        scores.sort(key=lambda x:x[1],reverse=True)

        return scores[:topk]